In [3]:
from pathlib import Path
from html.parser import HTMLParser
from urllib.parse import urljoin, urlsplit, unquote
from datetime import datetime, timezone
import hashlib
import io
import json
import re
import time

import requests
from pypdf import PdfReader

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")
SOURCE_PAGE = "https://www.commercega.gov/business/bid-opportunities"
SOLICITATION_ID = "commerce-ga-25-012"

RAW = PROJECT / "data/raw" / SOLICITATION_ID
MANIFESTS = PROJECT / "data/processed/manifests"

RAW.mkdir(parents=True, exist_ok=True)
MANIFESTS.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

session = requests.Session()
session.headers.update({
    "User-Agent": "GeorgiaBidReadiness/0.1 (academic document collection)"
})

response = session.get(SOURCE_PAGE, timeout=(15, 60))
response.raise_for_status()

snapshot = RAW / f"source_listing_{RUN_ID}.html"
snapshot.write_bytes(response.content)

class LinkCollector(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            href = dict(attrs).get("href")
            if href:
                self.links.append(urljoin(SOURCE_PAGE, href))

parser = LinkCollector()
parser.feed(response.text)

documents = []
seen_urls = set()

for url in parser.links:
    decoded_path = unquote(urlsplit(url).path)
    filename = decoded_path.rsplit("/", 1)[-1]

    # Limit collection to this specific solicitation folder.
    if "/25-012 Janitorial Services/" not in decoded_path:
        continue
    if not filename.lower().endswith(".pdf"):
        continue
    if url in seen_urls:
        continue
    seen_urls.add(url)

    amendment = re.search(r"ADDENDUM\s+(\d+)", filename, re.I)

    if amendment:
        number = int(amendment.group(1))
        doc_type = "addendum"
        document_id = f"{SOLICITATION_ID}-addendum-{number:02d}"
    elif (
        re.match(r"RFP\s+25-012\s*-\s*Janitorial Services", filename, re.I)
    ):
        number = 0
        doc_type = "original"
        document_id = f"{SOLICITATION_ID}-original"
    else:
        # Results and award notices are outside this initial packet.
        continue

    documents.append({
        "document_id": document_id,
        "solicitation_id": SOLICITATION_ID,
        "agency": "City of Commerce, Georgia",
        "document_type": doc_type,
        "amendment_number": number if number else None,
        "source_filename": filename,
        "source_url": url,
        "source_page": SOURCE_PAGE,
    })

documents.sort(key=lambda d: d["amendment_number"] or 0)

originals = [d for d in documents if d["document_type"] == "original"]
numbers = [
    d["amendment_number"]
    for d in documents
    if d["document_type"] == "addendum"
]

if len(originals) != 1 or sorted(numbers) != list(range(1, 12)):
    raise RuntimeError(
        "The source listing differs from the expected packet. "
        f"Found {len(originals)} originals and amendments {numbers}. "
        "Review the listing before downloading."
    )

print("Source listing saved:", snapshot)
print("Expected documents:", len(documents))

for document in documents:
    print(document["document_id"], "|", document["source_filename"])

Source listing saved: /home/jovyan/project/georgia-bid-readiness/data/raw/commerce-ga-25-012/source_listing_20261016T223117221997Z.html
Expected documents: 12
commerce-ga-25-012-original | RFP 25-012 - Janitorial Services City of Commerce .pdf
commerce-ga-25-012-addendum-01 | RFP 25-012 ADDENDUM 1- Janitorial Services City of Commerce.pdf
commerce-ga-25-012-addendum-02 | RFP 25-012 ADDENDUM 2- Janitorial Services City of Commerce.pdf
commerce-ga-25-012-addendum-03 | RFP 25-012 ADDENDUM 3- Janitorial Services City of Commerce.pdf
commerce-ga-25-012-addendum-04 | RFP 25-012 ADDENDUM 4- Janitorial Services City of Commerce.pdf
commerce-ga-25-012-addendum-05 | RFP 25-012 ADDENDUM 5- Janitorial Services City of Commerce.pdf
commerce-ga-25-012-addendum-06 | RFP 25-012 ADDENDUM 6- Janitorial Services City of Commerce.pdf
commerce-ga-25-012-addendum-07 | RFP 25-012 ADDENDUM 7- Janitorial Services City of Commerce.pdf
commerce-ga-25-012-addendum-08 | RFP 25-012 ADDENDUM 8- Janitorial Services C

In [4]:
manifest_path = MANIFESTS / f"{SOLICITATION_ID}_{RUN_ID}.json"

manifest = {
    "schema_version": 1,
    "ingestion_run_id": RUN_ID,
    "solicitation_id": SOLICITATION_ID,
    "source_page": SOURCE_PAGE,
    "source_listing_snapshot": snapshot.relative_to(PROJECT).as_posix(),
    "source_listing_sha256": hashlib.sha256(response.content).hexdigest(),
    "expected_document_count": 12,
    "packet_download_complete": False,
    "records": [],
}

def save_manifest():
    temporary = manifest_path.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )
    temporary.replace(manifest_path)

save_manifest()

for document in documents:
    record = {
        **document,
        "attempted_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "pending",
    }

    try:
        download = session.get(
            document["source_url"],
            timeout=(15, 90),
        )
        record["http_status"] = download.status_code
        record["resolved_url"] = download.url
        record["content_type"] = download.headers.get("Content-Type")
        download.raise_for_status()

        content = download.content

        if b"%PDF-" not in content[:1024]:
            raise ValueError("Response does not have a PDF header.")

        reader = PdfReader(io.BytesIO(content))
        if reader.is_encrypted:
            raise ValueError("Encrypted PDF requires separate review.")

        page_count = len(reader.pages)
        if page_count == 0:
            raise ValueError("PDF contains no pages.")

        digest = hashlib.sha256(content).hexdigest()

        # Full hash in the filename preserves distinct file versions.
        filename = f"{document['document_id']}__{digest}.pdf"
        destination = RAW / filename

        if destination.exists():
            existing_hash = hashlib.sha256(
                destination.read_bytes()
            ).hexdigest()

            if existing_hash != digest:
                raise ValueError("Stored file failed its integrity check.")

            storage_action = "reused_identical_file"
        else:
            temporary = destination.with_suffix(".part")
            temporary.write_bytes(content)
            temporary.replace(destination)
            storage_action = "saved_new_version"

        record.update({
            "status": "ok",
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "sha256": digest,
            "size_bytes": len(content),
            "page_count": page_count,
            "local_path": destination.relative_to(PROJECT).as_posix(),
            "storage_action": storage_action,
        })

        print(
            f"OK | {document['document_id']} | "
            f"{page_count} pages | {storage_action}",
            flush=True,
        )

    except Exception as error:
        record.update({
            "status": "failed",
            "error_type": type(error).__name__,
            "error": str(error),
        })
        print(
            f"FAILED | {document['document_id']} | {error}",
            flush=True,
        )

    manifest["records"].append(record)
    save_manifest()

    # Keep requests modest for the public website.
    time.sleep(1)

successful = [
    record for record in manifest["records"]
    if record["status"] == "ok"
]
failed = [
    record for record in manifest["records"]
    if record["status"] == "failed"
]

manifest["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
manifest["successful_document_count"] = len(successful)
manifest["failed_document_count"] = len(failed)
manifest["total_pages"] = sum(r["page_count"] for r in successful)
manifest["packet_download_complete"] = (
    len(successful) == manifest["expected_document_count"]
    and not failed
)
save_manifest()

print("\n--- INGESTION SUMMARY ---")
print("Successful PDFs:", len(successful))
print("Failed downloads:", len(failed))
print("Total pages:", manifest["total_pages"])
print("Expected packet downloaded:", manifest["packet_download_complete"])
print("Manifest:", manifest_path)

OK | commerce-ga-25-012-original | 15 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-01 | 1 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-02 | 1 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-03 | 1 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-04 | 1 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-05 | 2 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-06 | 1 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-07 | 1 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-08 | 1 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-09 | 1 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-10 | 2 pages | reused_identical_file
OK | commerce-ga-25-012-addendum-11 | 1 pages | reused_identical_file

--- INGESTION SUMMARY ---
Successful PDFs: 12
Failed downloads: 0
Total pages: 28
Expected packet downloaded: True
Manifest: /home/jovyan/project/georgia-bid-rea

In [5]:
pdf_files = sorted(RAW.glob("*.pdf"))

print("Stored PDF versions:", len(pdf_files))

actions = {}
for record in manifest["records"]:
    action = record.get("storage_action", record["status"])
    actions[action] = actions.get(action, 0) + 1

print("Latest run actions:", actions)

# Check every successfully registered file against its recorded hash.
integrity_errors = []

for record in manifest["records"]:
    if record["status"] != "ok":
        continue

    path = PROJECT / record["local_path"]

    if not path.is_file():
        integrity_errors.append(f"Missing: {path.name}")
        continue

    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()

    if actual_hash != record["sha256"]:
        integrity_errors.append(f"Hash mismatch: {path.name}")

print("Integrity errors:", integrity_errors)

Stored PDF versions: 12
Latest run actions: {'reused_identical_file': 12}
Integrity errors: []
